In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Imports
# ─────────────────────────────────────────────────────────────────────────────
import os, json, random, copy
from dataclasses import dataclass, asdict
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from tqdm import tqdm
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import KFold, train_test_split
 
print("Torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU  :", torch.cuda.get_device_name(0))

Torch: 2.3.1
CUDA : True
GPU  : NVIDIA GeForce GTX 1650


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Configuration
# ─────────────────────────────────────────────────────────────────────────────
@dataclass
class Config:

    # ── Embedding directories (Kaggle defaults; overridden below for local) ─
    # Kaggle dataset: wanianaeem/zenodo-pt-and-hm-dataset
    # Mounted at:     /kaggle/input/zenodo-pt-and-hm-dataset/
    #
    # To add QC-filtered scVI embeddings + spot_qc_mask.csv to the dataset:
    #   1. Go to kaggle.com/datasets/wanianaeem/zenodo-pt-and-hm-dataset
    #   2. Click "New Version" → drag in:
    #        • scvi_latent_pt_embeddings.zip  (zip the folder with the folder name included)
    #        • spot_qc_mask.csv
    #   3. Kaggle auto-extracts the zip → scvi_latent_pt_embeddings/ at dataset root
    #   4. Paths below are already correct for that layout — no edits needed.
    VISION_DIR:  str = "/kaggle/input/zenodo-pt-and-hm-dataset/Feature Extraction Embeddings/CONCH V1"
    GENE_DIR:    str = "/kaggle/input/zenodo-pt-and-hm-dataset/scvi_latent_pt_embeddings"
    CELL_DIR:    str = "/kaggle/input/zenodo-pt-and-hm-dataset/Cell Embedding Extraction/RCTD"
    output_dir:  str = "/kaggle/working/bridge_ckpts"

    # ── QC mask (built once by build_qc_mask.py) ─────────────────────────
    # All 20,395 patches stay on disk; only spots that pass MIN_GENES /
    # MIN_COUNTS are loaded at training time.  Change the thresholds here
    # to re-run with a different QC cut — no need to regenerate embeddings.
    QC_MASK_CSV: str = "/kaggle/input/zenodo-pt-and-hm-dataset/spot_qc_mask.csv"
    MIN_GENES:   int = 200    # nFeature_Spatial threshold
    MIN_COUNTS:  int = 400    # nCount_Spatial threshold

    # ── Cross-validation ──────────────────────────────────────────────────
    n_folds: int = 6

    # ── Model ─────────────────────────────────────────────────────────────
    proj_dim:    int   = 256
    dropout:     float = 0.30
    temperature: float = 0.07

    # ── Per-pair loss weights ─────────────────────────────────────────────
    weight_vg: float = 2.0
    weight_vc: float = 2.0
    weight_gc: float = 1.0

    # ── Training ─────────────────────────────────────────────────────────
    seed:               int   = 42
    epochs:             int   = 200
    batch_size:         int   = 512
    num_workers:        int   = 2
    lr:                 float = 3e-4
    weight_decay:       float = 5e-2
    patience:           int   = 10
    min_delta:          float = 1e-3
    warmup_epochs:      int   = 5
    grad_clip:          float = 1.0
    amp:                bool  = True
    accumulation_steps: int   = 8


cfg = Config()

# ── Local path overrides (auto-detected) ─────────────────────────────────────
IS_KAGGLE = os.path.exists('/kaggle/working')
if not IS_KAGGLE:
    _ROOT = Path.cwd().parent
    cfg.VISION_DIR   = str(_ROOT / "dataset/Feature Extraction Embeddings/CONCH V1")
    cfg.GENE_DIR     = str(_ROOT / "dataset/Gene Embedding Extraction/scvi_latent_pt_embeddings")
    cfg.CELL_DIR     = str(_ROOT / "dataset/Cell Embedding Extraction/RCTD")
    cfg.QC_MASK_CSV  = str(_ROOT / "Outputs/Patient-Sample-Information/spot_qc_mask.csv")
    cfg.output_dir   = str(_ROOT / "Outputs/bridge_ckpts")
    cfg.num_workers  = 0   # Windows multiprocessing: 0 is safest in a notebook

os.makedirs(cfg.output_dir, exist_ok=True)

print(f"Running  : {'Kaggle' if IS_KAGGLE else 'local'}")
print(f"VISION   : {cfg.VISION_DIR}")
print(f"GENE     : {cfg.GENE_DIR}")
print(f"CELL     : {cfg.CELL_DIR}")
print(f"QC mask  : {cfg.QC_MASK_CSV}")
print(f"QC thresh: nFeature>={cfg.MIN_GENES}, nCount>={cfg.MIN_COUNTS}")
print(f"Output   : {cfg.output_dir}")


def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device   :", device)

In [19]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Dataset
# ─────────────────────────────────────────────────────────────────────────────
def load_embedding_file(path: str) -> torch.Tensor:
    obj = torch.load(path, map_location="cpu")
    if isinstance(obj, dict):
        for key in ["embedding", "embeddings", "latent", "x", "z"]:
            if key in obj:
                obj = obj[key]
                break
        else:
            obj = next(v for v in obj.values() if torch.is_tensor(v))
    if isinstance(obj, np.ndarray):
        obj = torch.from_numpy(obj)
    return obj.float().reshape(-1)
 
 
class TriModalDataset(Dataset):
    def __init__(self, records: list):
        self.records = records
 
    def __len__(self):
        return len(self.records)
 
    def __getitem__(self, idx):
        row = self.records[idx]
        return {
            "id":     row["id"],
            "vision": row["vision"].float(),
            "gene":   load_embedding_file(row["gene_path"]),
            "cell":   load_embedding_file(row["cell_path"]),
        }

In [20]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Build aligned records  (with QC mask filtering)
#
# Vision .pt files are WSI-level bundles:
#   {"patient": str, "embeddings": Tensor[N, vision_dim], "patch_names": list[N]}
# Gene and cell .pt files are per-spot (one file per patch).
#
# Alignment = inner join on patch stem name across all three modalities,
# then filtered by the QC mask (cfg.MIN_GENES / cfg.MIN_COUNTS).
#
# To re-run with a different threshold (e.g. 300/600):
#   Change cfg.MIN_GENES and cfg.MIN_COUNTS in Cell 2 and re-run from Cell 4.
#   No embeddings need to be regenerated.
# ─────────────────────────────────────────────────────────────────────────────

# ── Load QC mask ──────────────────────────────────────────────────────────────
qc_dict = {}   # patch_stem -> (nFeature, nCount); empty = no extra QC filter
if cfg.QC_MASK_CSV and os.path.exists(cfg.QC_MASK_CSV):
    _qc_df = pd.read_csv(cfg.QC_MASK_CSV)
    qc_dict = {
        row["patch_stem"]: (int(row["nFeature"]), int(row["nCount"]))
        for _, row in _qc_df.iterrows()
    }
    _passing = sum(1 for nf, nc in qc_dict.values()
                   if nf >= cfg.MIN_GENES and nc >= cfg.MIN_COUNTS)
    print(f"QC mask loaded  : {len(qc_dict):,} patches total")
    print(f"Passes threshold: {_passing:,}  "
          f"(nFeature>={cfg.MIN_GENES}, nCount>={cfg.MIN_COUNTS})")
else:
    print("QC mask not found — no extra QC filter applied (modality inner join only).")

# ── Index per-spot gene and cell embedding files ──────────────────────────────
gene_dict = {f.stem: str(f) for f in sorted(Path(cfg.GENE_DIR).rglob("*.pt"))}
cell_dict = {f.stem: str(f) for f in sorted(Path(cfg.CELL_DIR).rglob("*.pt"))}
print(f"Gene .pt files  : {len(gene_dict):,}")
print(f"Cell .pt files  : {len(cell_dict):,}")

# ── Build aligned pairs ───────────────────────────────────────────────────────
pairs       = []
qc_rejected = 0

for vf in sorted(Path(cfg.VISION_DIR).rglob("*.pt")):
    slide_data  = torch.load(vf, map_location="cpu")
    patient     = slide_data["patient"]
    embeddings  = slide_data["embeddings"]    # Tensor [N_spots, vision_dim]
    patch_names = slide_data["patch_names"]   # list of N_spots patch stem strings

    matched = 0
    for idx, raw_name in enumerate(patch_names):
        name = Path(str(raw_name)).stem   # e.g. "IU_PDA_HM11_patch-000001_50_102"

        # Must have gene AND cell embeddings on disk
        if name not in gene_dict or name not in cell_dict:
            continue

        # QC mask: skip spots below nFeature/nCount threshold
        if qc_dict:
            qc = qc_dict.get(name)
            if qc is None or qc[0] < cfg.MIN_GENES or qc[1] < cfg.MIN_COUNTS:
                qc_rejected += 1
                continue

        pairs.append({
            "id":        name,
            "sample":    patient,
            "vision":    embeddings[idx],
            "gene_path": gene_dict[name],
            "cell_path": cell_dict[name],
        })
        matched += 1

    print(f"{patient}: {len(patch_names)} vision patches  "
          f"-> {matched} aligned  (vision_dim={embeddings.shape[1]})")

print(f"\nQC-rejected (below {cfg.MIN_GENES}/{cfg.MIN_COUNTS}): {qc_rejected:,}")
print(f"Total aligned spots: {len(pairs):,}")
assert len(pairs) > 0, "No aligned pairs found — check VISION_DIR / GENE_DIR / CELL_DIR paths."

QC mask loaded  : 20,395 patches total
Passes threshold: 18,859  (nFeature>=200, nCount>=400)
Gene .pt files  : 18,860
Cell .pt files  : 20,395
IU_PDA_HM11: 3931 vision patches  -> 3894 aligned  (vision_dim=512)
IU_PDA_HM13: 2182 vision patches  -> 1387 aligned  (vision_dim=512)
IU_PDA_T11: 2777 vision patches  -> 2677 aligned  (vision_dim=512)
IU_PDA_T1: 3530 vision patches  -> 3073 aligned  (vision_dim=512)
IU_PDA_T3: 4354 vision patches  -> 4241 aligned  (vision_dim=512)
IU_PDA_T4: 3621 vision patches  -> 3587 aligned  (vision_dim=512)

QC-rejected (below 200/400): 0
Total aligned spots: 18,859


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — Sample-level split  (6-fold LOSO + cohort-transfer test)
# ─────────────────────────────────────────────────────────────────────────────
#
# Strategy
# ─────────
# • 6-fold LOSO : each fold holds out exactly one sample
#   - Folds 1-2 : HM patients held out  (most scientifically critical —
#                 tests PT+one-HM training generalising to unseen HM)
#   - Folds 3-6 : PT patients held out
#   → Mean/std val loss across all 6 LOSO folds = reported CV performance
#   → HPO decisions are made on these 6 folds
#
# • Cohort-transfer (run ONCE after HPO locked):
#   - Train: all 4 PT samples  |  Val: both HM samples
#   → Tests alignment generalisation across the PT→HM cohort boundary
#   → De-facto test set for the paper
#
# IMPORTANT: SAMPLE_MAP keys must match the `patient` string inside your
#   vision .pt files (set during feature extraction).  The diagnostic block
#   below will warn loudly if there is a mismatch — check Cell 4 output.
# ─────────────────────────────────────────────────────────────────────────────

SAMPLE_MAP = {
    # .pt `patient` field  →  short label used in FOLD_DEFS
    # ── Hepatic Metastasis ─────────────────────────────────────────────────
    "IU_PDA_HM11": "HM11",
    "IU_PDA_HM13": "HM13",
    # ── Primary Tumour ─────────────────────────────────────────────────────
    "IU_PDA_T1":   "T1",
    "IU_PDA_T3":   "T3",
    "IU_PDA_T4":   "T4",
    "IU_PDA_T11":  "T11",
}

# Map every spot to its short sample label
spot_samples = np.array([
    SAMPLE_MAP.get(p["sample"], p["sample"]) for p in pairs
])

# Diagnostic: warn loudly if any patient string was not in SAMPLE_MAP
expected_labels = set(SAMPLE_MAP.values())
found_labels    = set(np.unique(spot_samples))
unmatched = found_labels - expected_labels
if unmatched:
    print("=" * 70)
    print("WARNING: The following sample IDs were NOT found in SAMPLE_MAP.")
    print("  All their spots will be excluded from every fold train AND val set!")
    print(f"  Unmatched IDs: {sorted(unmatched)}")
    print("  → Update SAMPLE_MAP keys to match the `patient` values printed in Cell 4.")
    print("=" * 70)
else:
    print("SAMPLE_MAP OK — all found patient IDs are recognised.")

# Per-sample spot counts
unique_samples, counts = np.unique(spot_samples, return_counts=True)
print("\nPer-sample spot counts:")
for s, c in zip(unique_samples, counts):
    print(f"  {s}: {c:,} spots")
print(f"  TOTAL: {len(pairs):,} spots")

# ── 6-fold LOSO definitions ──────────────────────────────────────────────────
FOLD_DEFS = [
    {"val": ["HM11"], "train": ["HM13", "T1",   "T3",  "T4",  "T11"]},  # Fold 1 — HM11 holdout ★
    {"val": ["HM13"], "train": ["HM11", "T1",   "T3",  "T4",  "T11"]},  # Fold 2 — HM13 holdout ★
    {"val": ["T1"],   "train": ["HM11", "HM13", "T3",  "T4",  "T11"]},  # Fold 3 — T1 holdout
    {"val": ["T3"],   "train": ["HM11", "HM13", "T1",  "T4",  "T11"]},  # Fold 4 — T3 holdout
    {"val": ["T4"],   "train": ["HM11", "HM13", "T1",  "T3",  "T11"]},  # Fold 5 — T4 holdout
    {"val": ["T11"],  "train": ["HM11", "HM13", "T1",  "T3",  "T4"]},   # Fold 6 — T11 holdout (T11+HM11 same patient)
]
# Cohort-transfer: train on ALL PT, test on ALL HM — run once after HPO locked
COHORT_FOLD_DEF = {
    "val":   ["HM11", "HM13"],
    "train": ["T1", "T3", "T4", "T11"],
}

DEV_FOLD_DEFS = FOLD_DEFS   # all 6 LOSO folds are development (HPO) folds

print("\nFold definitions:")
for i, fd in enumerate(DEV_FOLD_DEFS, 1):
    tr_mask = np.isin(spot_samples, fd["train"])
    vl_mask = np.isin(spot_samples, fd["val"])
    tag = " ★ HM holdout" if i <= 2 else (" [T11+HM11 same patient]" if i == 6 else "")
    print(f"  Fold {i} | train {sum(tr_mask):>5,} {fd['train']} "
          f"| val {sum(vl_mask):>5,} {fd['val']}{tag}")
ct_tr = np.isin(spot_samples, COHORT_FOLD_DEF["train"])
ct_vl = np.isin(spot_samples, COHORT_FOLD_DEF["val"])
print(f"  Cohort | train {sum(ct_tr):>5,} {COHORT_FOLD_DEF['train']} "
      f"| val {sum(ct_vl):>5,} {COHORT_FOLD_DEF['val']}  [run after HPO locked]")

# ── Build full dataset ────────────────────────────────────────────────────────
full_dataset = TriModalDataset(pairs)

# Infer embedding dims from first item
_sample_item = full_dataset[0]
vision_dim = _sample_item["vision"].shape[0]
gene_dim   = _sample_item["gene"].shape[0]
cell_dim   = _sample_item["cell"].shape[0]
print(f"\nDims — vision:{vision_dim}  gene:{gene_dim}  cell:{cell_dim}")

SAMPLE_MAP OK — all found patient IDs are recognised.

Per-sample spot counts:
  HM11: 3,894 spots
  HM13: 1,387 spots
  T1: 3,073 spots
  T11: 2,677 spots
  T3: 4,241 spots
  T4: 3,587 spots
  TOTAL: 18,859 spots

Fold definitions:
  Fold 1 | train 14,965 ['HM13', 'T1', 'T3', 'T4', 'T11'] | val 3,894 ['HM11'] ★ HM holdout
  Fold 2 | train 17,472 ['HM11', 'T1', 'T3', 'T4', 'T11'] | val 1,387 ['HM13'] ★ HM holdout
  Fold 3 | train 15,786 ['HM11', 'HM13', 'T3', 'T4', 'T11'] | val 3,073 ['T1']
  Fold 4 | train 14,618 ['HM11', 'HM13', 'T1', 'T4', 'T11'] | val 4,241 ['T3']
  Fold 5 | train 15,272 ['HM11', 'HM13', 'T1', 'T3', 'T11'] | val 3,587 ['T4']
  Fold 6 | train 16,182 ['HM11', 'HM13', 'T1', 'T3', 'T4'] | val 2,677 ['T11'] [T11+HM11 same patient]
  Cohort | train 13,578 ['T1', 'T3', 'T4', 'T11'] | val 5,281 ['HM11', 'HM13']  [run after HPO locked]

Dims — vision:512  gene:50  cell:15


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Model
# ─────────────────────────────────────────────────────────────────────────────
class MLPProjector(nn.Module):
    """
    MLP projector with configurable depth.
    - Vision (Pv): 3-layer  1536 → 768 → 384 → 256
      (extra layer gives Pv more capacity to find the biologically-relevant
       subspace within the high-dimensional UNI2-h / H-Optimus-0 space)
    - Gene  (Pg): 2-layer    50 → 256 → 256
    - Cell  (Pc): 2-layer    15 → 128 → 256
    All outputs are L2-normalised onto the unit hypersphere.
    """
    def __init__(self, input_dim: int, hidden_dims: list[int], proj_dim: int, dropout: float):
        super().__init__()
        layers = []
        in_d = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(in_d, h),
                nn.GELU(),
                nn.LayerNorm(h),
                nn.Dropout(dropout),
            ]
            in_d = h
        layers += [
            nn.Linear(in_d, proj_dim),
            nn.Dropout(dropout * 0.5),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return F.normalize(self.net(x), dim=-1)


class TriModalBridge(nn.Module):
    """
    Three independent projectors → shared 256-d unit-sphere space S.

    Pv : vision (1536d UNI2-h / 1152d H-Optimus-0)  →  3-layer  →  256d
    Pg : gene   (50d scVI latent)                    →  2-layer  →  256d
    Pc : cell   (15d RCTD proportions)               →  2-layer  →  256d

    Learnable per-pair log-temperatures replace the single global temperature.
    This lets the model independently calibrate how sharply to contrast each
    modality pair, which is important given the large difficulty gap between
    g↔c (easy) and v↔g / v↔c (harder).
    """
    def __init__(self, vision_dim: int, gene_dim: int, cell_dim: int, cfg: Config):
        super().__init__()
        j, d = cfg.proj_dim, cfg.dropout

        # Deeper Pv: 3 layers to find bio-relevant subspace in 1536-d vision
        self.Pv = MLPProjector(vision_dim, [768, 384], j, d)   # 1536→768→384→256
        self.Pg = MLPProjector(gene_dim,   [256],      j, d)   #   50→256→256
        self.Pc = MLPProjector(cell_dim,   [128],      j, d)   #   15→128→256

        # Learnable per-pair temperatures (initialised to cfg.temperature)
        # Clamped to [0.01, 0.5] during forward to prevent collapse / blow-up
        init_log_t = torch.log(torch.tensor(cfg.temperature))
        self.log_temp_vg = nn.Parameter(init_log_t.clone())
        self.log_temp_vc = nn.Parameter(init_log_t.clone())
        self.log_temp_gc = nn.Parameter(init_log_t.clone())

    def temperatures(self):
        """Return the current (clamped) per-pair temperatures as a dict."""
        return {
            "t_vg": self.log_temp_vg.clamp(-4.6, -0.7).exp().item(),  # [0.01, 0.5]
            "t_vc": self.log_temp_vc.clamp(-4.6, -0.7).exp().item(),
            "t_gc": self.log_temp_gc.clamp(-4.6, -0.7).exp().item(),
        }

    def forward(self, vision: torch.Tensor, gene: torch.Tensor, cell: torch.Tensor):
        return self.Pv(vision), self.Pg(gene), self.Pc(cell)

    def count_params(self):
        total = sum(p.numel() for p in self.parameters() if p.requires_grad)
        pv    = sum(p.numel() for p in self.Pv.parameters() if p.requires_grad)
        pg    = sum(p.numel() for p in self.Pg.parameters() if p.requires_grad)
        pc    = sum(p.numel() for p in self.Pc.parameters() if p.requires_grad)
        print(f"  Pv (3-layer): {pv:,}  |  Pg: {pg:,}  |  Pc: {pc:,}  |  Total: {total:,}")
        return total


def build_model():
    m = TriModalBridge(vision_dim, gene_dim, cell_dim, cfg).to(device)
    m.count_params()
    return m


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Loss
# ─────────────────────────────────────────────────────────────────────────────
def infonce(a: torch.Tensor, b: torch.Tensor, log_temp: nn.Parameter) -> torch.Tensor:
    """Symmetric InfoNCE with a learnable (clamped) temperature."""
    temp = log_temp.clamp(-4.6, -0.7).exp()          # [0.01, 0.5]
    logits = (a @ b.T) / temp
    labels = torch.arange(a.size(0), device=a.device)
    return 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))


def tri_modal_loss(zv, zg, zc, model: "TriModalBridge", cfg: "Config"):
    """
    Weighted sum of the three pairwise InfoNCE losses.
    Vision pairs (v↔g, v↔c) are up-weighted (×2 by default) to compensate
    for the larger alignment difficulty vs. g↔c.
    """
    l_vg = infonce(zv, zg, model.log_temp_vg)
    l_vc = infonce(zv, zc, model.log_temp_vc)
    l_gc = infonce(zg, zc, model.log_temp_gc)

    w_vg, w_vc, w_gc = cfg.weight_vg, cfg.weight_vc, cfg.weight_gc
    total_w = w_vg + w_vc + w_gc
    loss = (w_vg * l_vg + w_vc * l_vc + w_gc * l_gc) / total_w

    return loss, {"vg": l_vg.item(), "vc": l_vc.item(), "gc": l_gc.item()}


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Early stopping
# ─────────────────────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int, min_delta: float, path: str):
        self.patience   = patience
        self.min_delta  = min_delta
        self.path       = path
        self.best_loss  = float("inf")
        self.counter    = 0
        self.best_epoch = -1

    def step(self, val_loss: float, model: nn.Module, epoch: int) -> bool:
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss  = val_loss
            self.best_epoch = epoch
            self.counter    = 0
            torch.save(model.state_dict(), self.path)
            return False
        self.counter += 1
        return self.counter >= self.patience


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 9 — Single-epoch runner
# ─────────────────────────────────────────────────────────────────────────────
def run_epoch(model, loader, optimizer=None, scaler=None, train: bool = True):
    model.train(train)
    total_loss, total_n = 0.0, 0
    acc_steps = cfg.accumulation_steps if train else 1

    pbar = tqdm(loader, desc="Train" if train else "Val ", leave=False)
    for i, batch in enumerate(pbar):
        vision = batch["vision"].to(device, non_blocking=True)
        gene   = batch["gene"].to(device, non_blocking=True)
        cell   = batch["cell"].to(device, non_blocking=True)

        with torch.amp.autocast("cuda", enabled=(cfg.amp and device.type == "cuda")):
            zv, zg, zc = model(vision, gene, cell)
            loss, parts = tri_modal_loss(zv, zg, zc, model, cfg)   # pass model for learned temps

        if train:
            loss = loss / acc_steps
            scaler.scale(loss).backward()

            if (i + 1) % acc_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

        bs = vision.size(0)
        total_loss += loss.item() * bs * (acc_steps if train else 1)
        total_n    += bs

        temps = model.temperatures()
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            vg=f"{parts['vg']:.3f}",
            vc=f"{parts['vc']:.3f}",
            gc=f"{parts['gc']:.3f}",
            tvg=f"{temps['t_vg']:.3f}",
            tgc=f"{temps['t_gc']:.3f}",
        )

    return total_loss / total_n


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 10 — Evaluation helpers
# ─────────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def extract_projections(model, loader):
    model.eval()
    all_v, all_g, all_c, all_ids = [], [], [], []
    for batch in tqdm(loader, desc="Extract", leave=False):
        zv, zg, zc = model(
            batch["vision"].to(device),
            batch["gene"].to(device),
            batch["cell"].to(device),
        )
        all_v.append(zv.cpu()); all_g.append(zg.cpu()); all_c.append(zc.cpu())
        all_ids.extend(batch["id"])
    return torch.cat(all_v), torch.cat(all_g), torch.cat(all_c), all_ids


def recall_at_k(queries, gallery, k) -> float:
    k = min(k, gallery.size(0))
    topk = (queries @ gallery.T).topk(k, dim=1).indices
    target = torch.arange(queries.size(0)).unsqueeze(1)
    return (topk == target).any(dim=1).float().mean().item()


def recall_at_1(queries, gallery) -> float:
    return recall_at_k(queries, gallery, k=1)


def mean_reciprocal_rank(queries, gallery) -> float:
    """MRR: 1/rank of the correct match, averaged over all queries."""
    sims = queries @ gallery.T                      # [N, N]
    target = torch.arange(queries.size(0)).unsqueeze(1)
    # rank = position of the true match in descending-similarity order (1-indexed)
    order = sims.argsort(dim=1, descending=True)
    ranks = (order == target).float().argmax(dim=1) + 1  # [N]
    return (1.0 / ranks.float()).mean().item()


def retrieval_report(queries, gallery, ks=(1, 5, 10)) -> dict:
    out = {}
    for k in ks:
        out[f"R@{k}"] = recall_at_k(queries, gallery, k)
    out["MRR"] = mean_reciprocal_rank(queries, gallery)
    return out


def evaluate(model, loader, ks=(1, 5, 10)):
    """
    Full retrieval report for all 6 directed modality pairs.
    Returns a dict: {"v→g": {"R@1": ..., "R@5": ..., "R@10": ..., "MRR": ...}, ...}
    """
    V, G, C, _ = extract_projections(model, loader)
    pairs = {
        "v→g": (V, G), "g→v": (G, V),
        "v→c": (V, C), "c→v": (C, V),
        "g→c": (G, C), "c→g": (C, G),
    }
    results = {name: retrieval_report(q, g, ks) for name, (q, g) in pairs.items()}
    return results


def print_retrieval_report(results: dict, title: str = "RETRIEVAL"):
    print(f"\n── {title} ──────────")
    # header
    metric_names = list(next(iter(results.values())).keys())
    header = f"  {'pair':6s} " + " ".join(f"{m:>8s}" for m in metric_names)
    print(header)
    for pair, metrics in results.items():
        row = f"  {pair:6s} " + " ".join(f"{metrics[m]:8.4f}" for m in metric_names)
        print(row)


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11 — 6-Fold LOSO Cross-Validation  (all dev folds)
# ─────────────────────────────────────────────────────────────────────────────
#
# Folds 1-2 (HM holdout): each HM patient is left out once.
#   Training on PT + one HM; validates on the other HM.
#   → Most scientifically critical folds for metastatic generalisation.
#
# Folds 3-6 (PT holdout): each PT patient is left out once.
#   Training on both HM + remaining 3 PT; validates on the held-out PT.
#   → Ensures the model is not overfitting to a single PT patient.
#
# After all 6 folds: hyperparameters are LOCKED.
# Cohort-transfer test (Cell 11b) runs ONCE on those locked HPO settings.
# ─────────────────────────────────────────────────────────────────────────────

DEV_FOLDS = DEV_FOLD_DEFS   # 6 LOSO folds from Cell 5

# Storage
fold_train_histories = []
fold_val_histories   = []
fold_val_metrics     = []
fold_best_epochs     = []

loader_kw = dict(
    batch_size=cfg.batch_size, num_workers=cfg.num_workers, pin_memory=True
)


def make_warmup_cosine(optimizer, warmup_epochs, total_epochs, eta_min_ratio):
    cosine_epochs = max(total_epochs - warmup_epochs, 1)

    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / cosine_epochs
        cosine   = 0.5 * (1 + np.cos(np.pi * progress))
        return eta_min_ratio + (1 - eta_min_ratio) * cosine

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)


n_dev = len(DEV_FOLDS)

for fold, fold_def in enumerate(DEV_FOLDS, start=1):

    hm_tag = " ★ HM holdout" if fold <= 2 else ""
    print(f"\n{chr(9552)*60}")
    print(f"  FOLD {fold}/{n_dev}  |  val: {fold_def['val']}  train: {fold_def['train']}{hm_tag}")
    print(f"{chr(9552)*60}")

    fold_train_idx = np.where(np.isin(spot_samples, fold_def["train"]))[0]
    fold_val_idx   = np.where(np.isin(spot_samples, fold_def["val"]))[0]
    print(f"  spots — train: {len(fold_train_idx):,}  val: {len(fold_val_idx):,}")

    train_loader = DataLoader(
        Subset(full_dataset, fold_train_idx),
        shuffle=True, drop_last=True, **loader_kw
    )
    val_loader = DataLoader(
        Subset(full_dataset, fold_val_idx),
        shuffle=False, drop_last=False, **loader_kw
    )

    model     = build_model()
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
    )
    scheduler = make_warmup_cosine(
        optimizer, warmup_epochs=cfg.warmup_epochs,
        total_epochs=cfg.epochs, eta_min_ratio=0.01
    )
    scaler  = torch.amp.GradScaler("cuda", enabled=(cfg.amp and device.type == "cuda"))
    stopper = EarlyStopping(
        patience=cfg.patience,
        min_delta=cfg.min_delta,
        path=os.path.join(cfg.output_dir, f"best_fold{fold}.pt"),
    )

    train_hist, val_hist = [], []

    for epoch in range(1, cfg.epochs + 1):
        train_loss = run_epoch(model, train_loader, optimizer, scaler, train=True)
        val_loss   = run_epoch(model, val_loader,   train=False)
        scheduler.step()

        train_hist.append(train_loss)
        val_hist.append(val_loss)

        gap      = val_loss - train_loss
        improved = "✓" if val_loss < stopper.best_loss else " "
        print(
            f"  Ep {epoch:03d}/{cfg.epochs} | "
            f"train={train_loss:.4f} | val={val_loss:.4f} | "
            f"gap={gap:+.4f} | "
            f"lr={scheduler.get_last_lr()[0]:.2e}  {improved}"
        )

        if stopper.step(val_loss, model, epoch):
            print(f"\n  Early stop at epoch {epoch} "
                  f"(best ep {stopper.best_epoch}, val={stopper.best_loss:.4f})")
            break

    fold_train_histories.append(train_hist)
    fold_val_histories.append(val_hist)
    fold_val_metrics.append(stopper.best_loss)
    fold_best_epochs.append(stopper.best_epoch)

    print(f"\n  Fold {fold} best val loss : {stopper.best_loss:.4f} "
          f"@ epoch {stopper.best_epoch}")

# ── CV summary (all 6 LOSO folds) ────────────────────────────────────────────
print(f"\n{chr(9552)*60}")
print(f"  6-FOLD LOSO CV SUMMARY")
print(f"{chr(9552)*60}")

hm_losses = [fold_val_metrics[i] for i in range(2)]
pt_losses  = [fold_val_metrics[i] for i in range(2, 6)]

for i, (loss, ep) in enumerate(zip(fold_val_metrics, fold_best_epochs), 1):
    tag = " ★ HM holdout" if i <= 2 else (" [T11+HM11 same patient]" if i == 6 else "")
    print(f"  Fold {i}: best_val={loss:.4f}  best_epoch={ep:3d}  "
          f"val_sample={DEV_FOLDS[i-1]['val']}{tag}")

print(f"\n  All-fold  mean val loss : {np.mean(fold_val_metrics):.4f} ± {np.std(fold_val_metrics):.4f}")
print(f"  HM folds  mean val loss : {np.mean(hm_losses):.4f} ± {np.std(hm_losses):.4f}  (folds 1-2)")
print(f"  PT folds  mean val loss : {np.mean(pt_losses):.4f} ± {np.std(pt_losses):.4f}  (folds 3-6)")
print(f"  Mean best epoch         : {np.mean(fold_best_epochs):.1f}")
print()
print("  Hyperparameters are now LOCKED. Run Cell 11b for cohort-transfer test.")


════════════════════════════════════════════════════════════
  FOLD 1/6  |  val: ['HM11']  train: ['HM13', 'T1', 'T3', 'T4', 'T11'] ★ HM holdout
════════════════════════════════════════════════════════════
  spots — train: 14,965  val: 3,894
  Pv (3-layer): 790,144  |  Pg: 79,360  |  Pc: 35,328  |  Total: 904,835


  Ep 001/200 | train=6.4905 | val=6.2491 | gap=-0.2414 | lr=1.20e-04  ✓


  Ep 002/200 | train=6.4161 | val=6.1546 | gap=-0.2615 | lr=1.80e-04  ✓


  Ep 003/200 | train=6.3407 | val=6.0806 | gap=-0.2601 | lr=2.40e-04  ✓


  Ep 004/200 | train=6.2693 | val=6.0132 | gap=-0.2561 | lr=3.00e-04  ✓


...

  Ep 009/200 | train=5.7930 | val=6.1282 | gap=+0.3353 | lr=3.00e-04   


  Ep 010/200 | train=5.7417 | val=6.1103 | gap=+0.3686 | lr=3.00e-04   


  Ep 011/200 | train=5.7022 | val=6.1377 | gap=+0.4355 | lr=2.99e-04   


  Ep 012/200 | train=5.6604 | val=6.1767 | gap=+0.5163 | lr=2.99e-04   


KeyboardInterrupt: 

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 11b — Cohort-Transfer Test  (run ONCE after all 6 LOSO folds are done)
# ─────────────────────────────────────────────────────────────────────────────
#
# Train : T1, T3, T4, T11  (all primary tumour)
# Val   : HM11, HM13       (all hepatic metastasis — never in LOSO training)
#
# Purpose: tests whether Phase A alignment generalises across the PT→HM
#          cohort boundary with NO HM data seen during this training run.
#          These are the de-facto test numbers for the paper.
#
# • Training length = mean best epoch from all 6 LOSO folds (no early stopping —
#   HPO decisions are locked above)
# • Do NOT use these metrics to adjust any hyperparameter
# ─────────────────────────────────────────────────────────────────────────────

cohort_epochs = int(np.round(np.mean(fold_best_epochs)))
print(f"\n{chr(9552)*60}")
print(f"  COHORT-TRANSFER TEST  (de-facto held-out test — PT→HM)")
print(f"  train: {COHORT_FOLD_DEF['train']}  |  val: {COHORT_FOLD_DEF['val']}")
print(f"  Fixed epochs: {cohort_epochs}  (mean best epoch from 6 LOSO folds)")
print(f"  *** NO early stopping — hyperparameters are locked ***")
print(f"{chr(9552)*60}")

cohort_train_idx = np.where(np.isin(spot_samples, COHORT_FOLD_DEF["train"]))[0]
cohort_val_idx   = np.where(np.isin(spot_samples, COHORT_FOLD_DEF["val"]))[0]
print(f"  spots — train: {len(cohort_train_idx):,}  val: {len(cohort_val_idx):,}")

cohort_train_loader = DataLoader(
    Subset(full_dataset, cohort_train_idx),
    shuffle=True, drop_last=True, **loader_kw
)
cohort_val_loader = DataLoader(
    Subset(full_dataset, cohort_val_idx),
    shuffle=False, drop_last=False, **loader_kw
)

cohort_model     = build_model()
cohort_optimizer = torch.optim.AdamW(
    cohort_model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
)
cohort_scheduler = make_warmup_cosine(
    cohort_optimizer,
    warmup_epochs=min(cfg.warmup_epochs, max(cohort_epochs - 1, 1)),
    total_epochs=cohort_epochs,
    eta_min_ratio=0.01,
)
cohort_scaler = torch.amp.GradScaler("cuda", enabled=(cfg.amp and device.type == "cuda"))

cohort_train_hist, cohort_val_hist = [], []

for epoch in range(1, cohort_epochs + 1):
    train_loss = run_epoch(cohort_model, cohort_train_loader, cohort_optimizer,
                           cohort_scaler, train=True)
    val_loss   = run_epoch(cohort_model, cohort_val_loader, train=False)
    cohort_scheduler.step()

    cohort_train_hist.append(train_loss)
    cohort_val_hist.append(val_loss)

    print(
        f"  Ep {epoch:03d}/{cohort_epochs} | "
        f"train={train_loss:.4f} | val={val_loss:.4f} | "
        f"gap={val_loss - train_loss:+.4f} | "
        f"lr={cohort_scheduler.get_last_lr()[0]:.2e}"
    )

# Save checkpoint
torch.save(
    cohort_model.state_dict(),
    os.path.join(cfg.output_dir, "best_fold_cohort.pt")
)

cohort_val_loss = cohort_val_hist[-1]

# Full retrieval report on HM spots
cohort_metrics = evaluate(cohort_model, cohort_val_loader)
print_retrieval_report(cohort_metrics, title="COHORT-TRANSFER — HM RETRIEVAL (v→g is the key metric)")

# ── Domain gap vs LOSO baseline ───────────────────────────────────────────────
loso_mean = np.mean(fold_val_metrics)
hm_loso_mean = np.mean([fold_val_metrics[0], fold_val_metrics[1]])
domain_gap = cohort_val_loss - loso_mean
print(f"\n  Domain gap summary")
print(f"  6-fold LOSO mean val loss      : {loso_mean:.4f}")
print(f"  HM-holdout folds (1-2) mean    : {hm_loso_mean:.4f}")
print(f"  Cohort-transfer val loss (HM)  : {cohort_val_loss:.4f}")
print(f"  Δ vs LOSO mean                 : {domain_gap:+.4f}")
if domain_gap > 0.05:
    print("  ↑ Notable cohort gap — representations are partially PT-specific.")
elif domain_gap > 0:
    print("  ↑ Small cohort gap — alignment generalises reasonably.")
else:
    print("  ↓ No cohort gap — HM val loss ≤ LOSO mean (strong generalisation).")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 12 — Training curves: all 6 LOSO folds
# ─────────────────────────────────────────────────────────────────────────────
max_epochs = max(len(h) for h in fold_train_histories)


def pad(hist, length):
    return hist + [hist[-1]] * (length - len(hist))


train_mat = np.array([pad(h, max_epochs) for h in fold_train_histories])
val_mat   = np.array([pad(h, max_epochs) for h in fold_val_histories])

epochs_x = np.arange(1, max_epochs + 1)

fig, ax = plt.subplots(figsize=(12, 6), facecolor="white")

# 6 distinct colors — folds 1-2 (HM holdout) use warm tones, folds 3-6 use cool tones
FOLD_COLORS = ["#E63946", "#FF6B6B", "#4C72B0", "#55A868", "#DD8452", "#8172B2"]
FOLD_LABELS = ["Fold 1 — HM11 ★", "Fold 2 — HM13 ★",
               "Fold 3 — T1",    "Fold 4 — T3",
               "Fold 5 — T4",    "Fold 6 — T11"]

for fold_i in range(len(DEV_FOLDS)):
    n_actual = len(fold_train_histories[fold_i])
    col      = FOLD_COLORS[fold_i]
    lbl      = FOLD_LABELS[fold_i]
    val_best = fold_val_metrics[fold_i]

    ax.plot(
        epochs_x[:n_actual], train_mat[fold_i, :n_actual],
        color=col, alpha=0.20, linewidth=1.0, linestyle="--"
    )
    ax.plot(
        epochs_x[:n_actual], val_mat[fold_i, :n_actual],
        color=col, alpha=0.55, linewidth=1.3,
        label=f"{lbl}  val={val_best:.3f}"
    )
    best_ep = fold_best_epochs[fold_i]
    ax.axvline(best_ep, color=col, linestyle=":", alpha=0.35, linewidth=1.0)

# Mean ± std band
mean_train = train_mat.mean(axis=0)
mean_val   = val_mat.mean(axis=0)
std_val    = val_mat.std(axis=0)
std_train  = train_mat.std(axis=0)

ax.plot(epochs_x, mean_train, color="#333333", linewidth=2.0,
        linestyle="--", label="Mean train (all 6 folds)")
ax.plot(epochs_x, mean_val,   color="#1A1A2E", linewidth=2.5,
        label=f"Mean val (all 6 folds)  {np.mean(fold_val_metrics):.3f}±{np.std(fold_val_metrics):.3f}")
ax.fill_between(epochs_x,
                mean_val - std_val, mean_val + std_val,
                color="#1A1A2E", alpha=0.10, label="Val ± 1 std")
ax.fill_between(epochs_x,
                mean_train - std_train, mean_train + std_train,
                color="#333333", alpha=0.06, label="Train ± 1 std")

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("InfoNCE Loss", fontsize=12)
ax.set_title("6-Fold LOSO Cross-Validation — TriModalBridge Training Curve\n"
             "(★ = HM holdout folds — most critical for metastatic generalisation)", fontsize=12)
ax.legend(fontsize=7.5, loc="upper right", ncol=2)
ax.grid(True, alpha=0.3)
ax.set_facecolor("white")
ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
fig.tight_layout()
fig.savefig(
    os.path.join(cfg.output_dir, "loso_training_curve.png"),
    dpi=300, bbox_inches="tight", facecolor="white"
)
plt.show()
print("Training curve saved.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 13 — Final model: retrain on ALL 6 samples for Phase B
# ─────────────────────────────────────────────────────────────────────────────
# Use mean best epoch from all 6 LOSO folds as the fixed training length.
# This model sees ALL spots and produces the joint-space embeddings that
# Phase B (pt-hm-mlp-head.ipynb) uses to compute the direction vector.
# ─────────────────────────────────────────────────────────────────────────────
final_epochs = int(np.round(np.mean(fold_best_epochs)))
print(f"\nRetraining final model for {final_epochs} epochs on all 6 samples …")

final_loader = DataLoader(
    Subset(full_dataset, np.arange(len(pairs))),
    shuffle=True, drop_last=True, **loader_kw
)

final_model     = build_model()
final_optimizer = torch.optim.AdamW(
    final_model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay
)
final_scheduler = make_warmup_cosine(
    final_optimizer,
    warmup_epochs=min(cfg.warmup_epochs, max(final_epochs - 1, 1)),
    total_epochs=final_epochs,
    eta_min_ratio=0.01,
)
final_scaler = torch.amp.GradScaler("cuda", enabled=(cfg.amp and device.type == "cuda"))

for epoch in range(1, final_epochs + 1):
    loss = run_epoch(final_model, final_loader, final_optimizer, final_scaler, train=True)
    final_scheduler.step()
    print(f"  Final ep {epoch:03d}/{final_epochs} | train={loss:.4f} | "
          f"lr={final_scheduler.get_last_lr()[0]:.2e}")

torch.save(final_model.state_dict(),
           os.path.join(cfg.output_dir, "final_model.pt"))
print(f"\nFinal model saved → {cfg.output_dir}/final_model.pt")
print("Cohort-transfer metrics from Cell 11b are the paper's test numbers.")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 14 — Save all artefacts
# ─────────────────────────────────────────────────────────────────────────────
n_dev_folds = len(DEV_FOLD_DEFS)

cv_summary = {
    # 6-fold LOSO — patient-level generalisation metrics
    "n_loso_folds":              n_dev_folds,
    "fold_best_val_losses":      fold_val_metrics,          # list of 6
    "fold_best_epochs":          fold_best_epochs,          # list of 6
    "fold_val_samples":          [fd["val"] for fd in DEV_FOLD_DEFS],
    "mean_val_loss_all":         float(np.mean(fold_val_metrics)),
    "std_val_loss_all":          float(np.std(fold_val_metrics)),
    "mean_val_loss_hm_folds":    float(np.mean([fold_val_metrics[0], fold_val_metrics[1]])),
    "mean_val_loss_pt_folds":    float(np.mean(fold_val_metrics[2:])),
    "final_train_epochs":        final_epochs,
    # Cohort-transfer test — separate from LOSO mean
    "cohort_transfer_val_loss":  float(cohort_val_loss),
    "cohort_transfer_metrics":   cohort_metrics,
    "cohort_domain_gap":         float(cohort_val_loss - np.mean(fold_val_metrics)),
}

with open(os.path.join(cfg.output_dir, "cv_summary.json"), "w") as f:
    json.dump(cv_summary, f, indent=2)

with open(os.path.join(cfg.output_dir, "config.json"), "w") as f:
    json.dump(asdict(cfg), f, indent=2)

# ── Pad histories and save CSV ────────────────────────────────────────────────
def pad(hist, length):
    return hist + [hist[-1]] * (length - len(hist))

max_ep = max(len(h) for h in fold_train_histories)

pd.DataFrame(
    {f"fold{i+1}_train": pad(fold_train_histories[i], max_ep) for i in range(n_dev_folds)}
  | {f"fold{i+1}_val":   pad(fold_val_histories[i],   max_ep) for i in range(n_dev_folds)}
).to_csv(os.path.join(cfg.output_dir, "loso_history.csv"), index=False)

print("All artefacts saved to:", cfg.output_dir)
print(f"  cv_summary.json  ({n_dev_folds}-fold LOSO + cohort-transfer)")
print(f"  loso_history.csv  ({max_ep} epochs × {n_dev_folds} folds × 2 splits)")

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 15 — Project ALL spots with final model → save embeddings for Phase B
# ─────────────────────────────────────────────────────────────────────────────
#
# Phase B (pt-hm-mlp-head.ipynb) needs joint-space embeddings for every spot
# across all 6 samples so it can:
#   1. Compute the metastatic direction vector Δ_meta = μ_HM − μ_PT
#   2. Score each PT spot by cosine similarity to Δ_meta
#   3. Train the SpatialGAT for spatial smoothing
#
# We project ALL spots with the final model (trained on all 6 samples).
# ─────────────────────────────────────────────────────────────────────────────

@torch.no_grad()
def run_inference(model_path: str, loader: DataLoader):
    model = TriModalBridge(vision_dim, gene_dim, cell_dim, cfg).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    V, G, C, ids = extract_projections(model, loader)
    return {"ids": ids, "vision_emb": V, "gene_emb": G, "cell_emb": C}, model


# Full dataset loader — projects every spot from all 6 samples
all_spots_loader = DataLoader(
    full_dataset,
    batch_size=cfg.batch_size,
    num_workers=cfg.num_workers,
    pin_memory=True,
    shuffle=False,
    drop_last=False,
)

inference_ckpt = os.path.join(cfg.output_dir, "final_model.pt")
inf_results, inf_model = run_inference(inference_ckpt, all_spots_loader)

print(f"Inference complete — {len(inf_results['ids']):,} spots projected")
print(f"  vision_emb : {inf_results['vision_emb'].shape}")
print(f"  gene_emb   : {inf_results['gene_emb'].shape}")
print(f"  cell_emb   : {inf_results['cell_emb'].shape}")

# Retrieval metrics on the full set (sanity check on final model quality)
inf_metrics = {
    name: retrieval_report(q, g)
    for name, (q, g) in {
        "v→g": (inf_results["vision_emb"], inf_results["gene_emb"]),
        "g→v": (inf_results["gene_emb"],   inf_results["vision_emb"]),
        "v→c": (inf_results["vision_emb"], inf_results["cell_emb"]),
        "c→v": (inf_results["cell_emb"],   inf_results["vision_emb"]),
        "g→c": (inf_results["gene_emb"],   inf_results["cell_emb"]),
        "c→g": (inf_results["cell_emb"],   inf_results["gene_emb"]),
    }.items()
}
print_retrieval_report(inf_metrics, title="FINAL MODEL — ALL-SPOT RETRIEVAL (sanity check)")

# Save per-spot embeddings + sample labels for Phase B
torch.save(
    {
        "ids":          inf_results["ids"],
        "vision_emb":   inf_results["vision_emb"],   # [N, 256] — Phase B uses vision only at inference
        "gene_emb":     inf_results["gene_emb"],     # [N, 256]
        "cell_emb":     inf_results["cell_emb"],     # [N, 256]
        "spot_samples": list(spot_samples),          # sample label per spot (HM11/HM13/T1/...)
    },
    os.path.join(cfg.output_dir, "all_spots_embeddings.pt"),
)
print("\nSaved → all_spots_embeddings.pt  (input for Phase B — pt-hm-mlp-head.ipynb)")


# ── Single-spot helper (for debugging / interactive use) ─────────────────────
@torch.no_grad()
def embed_single_spot(model, vision_vec: torch.Tensor,
                      gene_vec: torch.Tensor, cell_vec: torch.Tensor):
    model.eval()
    v = vision_vec.float().unsqueeze(0).to(device)
    g = gene_vec.float().unsqueeze(0).to(device)
    c = cell_vec.float().unsqueeze(0).to(device)
    return model(v, g, c)

# Example (uncomment to test):
# _s = full_dataset[0]
# zv, zg, zc = embed_single_spot(inf_model, _s["vision"], _s["gene"], _s["cell"])
# print(zv.shape, zg.shape, zc.shape)